In [137]:
import json
import glob
import shutil
import os
import re
import os.path as op
import requests
# base_url = "http://192.168.31.188:4094/ZWXK_AI/API/task" # test server
# base_url = "http://172.18.101.171:10000/ZWXK_AI/API/task" # test server
# base_url = "http://192.168.31.188:3888/ZWXK_AI/API/task" # test server
# base_url = "http://172.18.100.163:3090/ZWXK_AI/API/image_matching" # local
# base_url = "http://172.18.167.133:3090/ZWXK_AI/API/task" # local
base_url = "http://172.18.167.133:3889/api/v1/smear_analysis" # local

In [138]:
def convert_output(output):
    return [
        dict(
            row_index=row,
            col_index=col,
            image_path=op.abspath(data[(row, col)]),
            x_topleft=int(x),
            y_topleft=int(y),
            residual=residual,
            component_id=component_id
        )
        for (row, col), ((x, y), residual, component_id) in output.items()
    ]


# root = r"\\192.168.31.180\共享目录-软件部\拼图提升40倍图片样本\带平台坐标预估的拼图坐标\data2025063002"
# root = r"D:\data\data2025070117"
root = r"D:\data\data20250901"
path = op.join(root, "Images")
images = glob.glob(op.join(path, "*.jpg"))
pattern = re.compile(r"Pos\[(\d+)\]\[(\d+)\]")
rows = [int(pattern.findall(x)[0][1]) for x in images]
cols = [int(pattern.findall(x)[0][0]) for x in images]
data = list(zip(rows, cols, images))
data = [
    (row, col, img)
    for row, col, img in data
]
data.sort(key=lambda x: (x[0], x[1]))
num_rows = max([x[0] for x in data]) + 1
# num_rows = 2
num_cols = max([x[1] for x in data]) + 1
# num_cols = 2

test_json = json.load(open(r'D:\data\zwxk-algorithm-backend\test.json', encoding='utf-8'))
map_dict = {}
for one in test_json:
    map_dict[(one['row_index'], one['col_index'])] = (one['x_topleft'], one['y_topleft'])

imageinfo = op.join(root, "picInfo.json")
with open(imageinfo, 'r') as f:
    image_info = json.load(f)
    info = image_info.copy()
    info.pop('base_pixel', None)
    image_info = {
        (x['index_y'], x['index_x']): [
            int(x['left_x']),
            int(x['left_y']),
            int(x['top_x']),
            int(x['top_y']),
        ]
        for x in image_info['base_pixel']
    }

data = {
    (x[0], x[1]): x[2] for x in data
}

In [139]:
url = f"{base_url}/create_task"

d = {
    "num_rows": num_rows,
    "num_cols": num_cols,
    "tile_width": info['pixel_width'],
    "tile_height": info['pixel_height'],
    'smear_type': "BM",
    'dpi': 40
}

# post to url
response = requests.post(url, json=d)
if response.status_code != 200:
    raise Exception(f"Failed to create task: {response.text}")

json_data = response.json()

print(json_data)
task_id = json_data['task_id']

{'ret_code': 200, 'ret_desc': '接口调用成功', 'task_id': '2a86a9b5d6e14d808676e955aa9e8228'}


In [143]:
get_result_url = f"{base_url}/check_task_status"
params = {
    "task_id": task_id,
}
response = requests.post(get_result_url, json=params)
# if response.status_code != 200:
#     raise Exception(f"Failed to get result: {response.text}")
print(response.json())

{'ret_code': 200, 'ret_desc': '接口调用成功', 'task_status': '任务已完成'}


In [142]:
get_result_url = f"{base_url}/get_task_result"
params = {
    "task_id": task_id,
    'roi_xmax': 1,
    'roi_ymax': 1,
}
response = requests.post(get_result_url, json=params)
if response.status_code != 200:
    raise Exception(f"Failed to get result: {response.text}")
print(response.json())
if response.json().get('match_result'):
    print(response.json())

{'cell_count': 0, 'cell_list': [], 'ret_code': 200, 'ret_desc': '接口调用成功'}


In [141]:
check_image_url = f"{base_url}/check_missing_tiles"
params = {
    "task_id": task_id,
    'row_id': num_rows,
    'max_ret_num': 100,
}
response = requests.post(check_image_url, json=params)
if response.status_code != 200:
    raise Exception(f"Failed to check image: {response.text}")
print(response.json())

{'missing_tiles': [], 'ret_code': 200, 'ret_desc': '接口调用成功'}


In [ ]:
check_image_url = f"{base_url}/get_node"
params = {
    "task_id": task_id,
}
response = requests.get(check_image_url, params=params)
print(response.json()['edges'])

In [140]:
import time
import requests
import concurrent.futures
from tqdm import tqdm
import os
# session = requests.Session()

# 上传单个文件的函数
def upload_file(item):
    row, col = item
    if row >= num_rows or col >= num_cols:
        return

    path = data[(row, col)]
    position_x, position_y = map_dict[(row, col)]
    img_info = image_info.get((row, col), [0, 0, 0, 0])

    form_data = {
        "task_id": task_id,
        "row_index": row,
        "col_index": col,
        "position_x": position_x,
        "position_y": position_y,
    }

    try:
        with open(path, 'rb') as f:
            imgdata = f.read()

        files = {
            "tile_image": (os.path.basename(path), imgdata)
        }

        response = requests.post(
            f"{base_url}/upload_tile",
            data=form_data, files=files, timeout=5
        )

        # 可以根据需要返回结果或日志
        return response.json()
        # return 200
    except Exception as e:
        return f"Error: {e}"

# 多线程执行上传
def run_uploads_parallel(items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:  # 控制并发数
        futures = [executor.submit(upload_file, item) for item in items]

        # 使用 tqdm 显示进度条
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Uploading"):
            # print(future.result())  # 打印每个上传的结果
            pass

        # 如果需要处理结果，可以使用 futures 或 future.result()

def run_uploads_sequential(items):
    for one in items:
        upload_file(one)

run_uploads_parallel(data)
# run_uploads_sequential(data)

Uploading: 100%|██████████| 575/575 [00:08<00:00, 69.53it/s]


In [ ]:
get_result_url = f"{base_url}/get_result"
params = {
    "task_id": task_id,
}
response = requests.post(get_result_url, params=params)
if response.status_code != 200:
    raise Exception(f"Failed to get result: {response.text}")
output = response.json()

result = [
    dict(
        image_path=op.abspath(data[(x['row_index'], x['col_index'])]),
        row_index=x['row_index'],
        col_index=x['col_index'],
        x_topleft=int(x['x_topleft']),
        y_topleft=int(x['y_topleft']),
        residual=0,
        component_id=0,
    )
    for x in output['match_result']
]
with open(r"test.json", 'w') as f:
    json.dump(result, f, indent=4)

In [ ]:
get_result_url = f"{base_url}/get_task_result_x40"
params = {"task_id": task_id}
response = requests.post(get_result_url, params=params)
if response.status_code != 200:
    raise Exception(f"Failed to get task result: {response.text}")
predictions = response.json()['match_result']